# Temel Görüntü İşleme\nPiksel, renk uzayları, histogram, filtreleme, kenar tespiti, morfoloji ve kontur bulma.\n\nscipy built-in görsellerle çalışır. Kendi İstanbul fotoğraflarını `fotograflar/` klasörüne ekleyip ilgili hücrede yolunu değiştirerek kullanabilirsin.

In [ ]:
import cv2\nimport numpy as np\nimport matplotlib.pyplot as plt\nfrom scipy import datasets\nimport os\n\nos.makedirs('cikti', exist_ok=True)\nplt.rcParams['figure.figsize'] = (12, 8)\nprint('Kütüphaneler hazır.')

## 1. Görüntü Yükleme ve Piksel Analizi

In [ ]:
# scipy built-in rakun fotoğrafı (768x1024, RGB)\n# Kendi fotoğrafın için: img = cv2.imread('fotograflar/istanbul.jpg'); img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)\nimg = datasets.face()\nh, w, c = img.shape\nprint(f'Boyut: {h}x{w}, Kanal: {c}')\nprint(f'Veri tipi: {img.dtype}, Min: {img.min()}, Max: {img.max()}, Ort: {img.mean():.1f}')\n\nplt.imshow(img)\nplt.title('Orijinal Görüntü (Rakun)')\nplt.axis('off')\nplt.show()

## 2. Renk Uzayları (RGB, Grayscale, HSV)

In [ ]:
gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)\nhsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)\n\nfig, axes = plt.subplots(1, 3)\naxes[0].imshow(img); axes[0].set_title('RGB')\naxes[1].imshow(gray, cmap='gray'); axes[1].set_title('Grayscale')\naxes[2].imshow(hsv); axes[2].set_title('HSV')\nfor ax in axes: ax.axis('off')\nplt.savefig('cikti/renk_uzaylari.png', dpi=100, bbox_inches='tight')\nplt.show()

## 3. Histogram Analizi ve Eşitleme

In [ ]:
eq = cv2.equalizeHist(gray)\n\nfig, axes = plt.subplots(2, 2)\naxes[0, 0].imshow(gray, cmap='gray'); axes[0, 0].set_title('Orijinal Gri')\naxes[0, 1].hist(gray.ravel(), 256, [0, 256], color='black', alpha=0.7); axes[0, 1].set_title('Histogram')\naxes[1, 0].imshow(eq, cmap='gray'); axes[1, 0].set_title('Eşitlenmiş')\naxes[1, 1].hist(eq.ravel(), 256, [0, 256], color='black', alpha=0.7); axes[1, 1].set_title('Eşitlenmiş Histogram')\nfor row in axes:\n    for ax in row:\n        if ax not in [axes[0, 1], axes[1, 1]]: ax.axis('off')\nplt.savefig('cikti/histogram.png', dpi=100, bbox_inches='tight')\nplt.show()

## 4. Filtreleme (Bulanıklaştırma ve Keskinleştirme)

In [ ]:
blur = cv2.GaussianBlur(gray, (15, 15), 0)\nkernel = np.array([[-1, -1, -1], [-1, 9, -1], [-1, -1, -1]])\nsharpen = cv2.filter2D(gray, -1, kernel)\n\nfig, axes = plt.subplots(1, 3)\naxes[0].imshow(gray, cmap='gray'); axes[0].set_title('Orijinal')\naxes[1].imshow(blur, cmap='gray'); axes[1].set_title('Gaussian Blur')\naxes[2].imshow(sharpen, cmap='gray'); axes[2].set_title('Keskinleştirme')\nfor ax in axes: ax.axis('off')\nplt.savefig('cikti/filtreler.png', dpi=100, bbox_inches='tight')\nplt.show()

## 5. Kenar Tespiti (Canny ve Sobel)

In [ ]:
canny = cv2.Canny(gray, 80, 200)\nsobel_x = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)\nsobel_y = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)\nsobel = np.uint8(np.clip(np.sqrt(sobel_x**2 + sobel_y**2), 0, 255))\n\nfig, axes = plt.subplots(1, 3)\naxes[0].imshow(gray, cmap='gray'); axes[0].set_title('Orijinal')\naxes[1].imshow(canny, cmap='gray'); axes[1].set_title('Canny')\naxes[2].imshow(sobel, cmap='gray'); axes[2].set_title('Sobel')\nfor ax in axes: ax.axis('off')\nplt.savefig('cikti/kenar.png', dpi=100, bbox_inches='tight')\nplt.show()

## 6. Morfolojik İşlemler

In [ ]:
_, binary = cv2.threshold(gray, 100, 255, cv2.THRESH_BINARY)\nkernel = np.ones((5, 5), np.uint8)\n\nerode = cv2.erode(binary, kernel)\ndilate = cv2.dilate(binary, kernel)\nopening = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel)\nclosing = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel)\n\nfig, axes = plt.subplots(2, 3)\nfor ax, title, im in zip(axes.flat, ['Binary', 'Erozyon', 'Genişleme', '', 'Açma', 'Kapama'], [binary, erode, dilate, np.zeros_like(binary), opening, closing]):\n    if im.any(): ax.imshow(im, cmap='gray')\n    ax.set_title(title); ax.axis('off')\nplt.savefig('cikti/morfoloji.png', dpi=100, bbox_inches='tight')\nplt.show()

## 7. Kontur Bulma

In [ ]:
konturlar, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)\nkontur_resim = img.copy()\ncv2.drawContours(kontur_resim, konturlar, -1, (255, 255, 0), 2)\n\nfig, axes = plt.subplots(1, 2)\naxes[0].imshow(gray, cmap='gray'); axes[0].set_title('Orijinal')\naxes[1].imshow(kontur_resim); axes[1].set_title(f'{len(konturlar)} Kontur')\nfor ax in axes: ax.axis('off')\nplt.savefig('cikti/kontur.png', dpi=100, bbox_inches='tight')\nplt.show()\nprint(f'İşlem tamamlandı. Çıktılar cikti/ klasöründe.')